# Chronos-2 on a Real Multivariate Dataset: Jena Climate

This notebook is a companion to `Chronos2_Zero_to_Mastery.ipynb`. It focuses on **genuinely multivariate** forecasting using a real sensor dataset with many correlated channels: **Jena Climate** (weather station measurements).

After running this notebook, you will be able to:

- Load Jena Climate and build a leakage-safe rolling backtest
- Use Chronos-2's **native multivariate target** format (multiple channels forecast jointly)
- Add **known-future calendar covariates** via the dict input schema
- Benchmark Chronos-2 vs honest baselines (naive, seasonal-naive) with MAE and sMAPE

## Runtime toggle

- `CHRONOS2_TUTORIAL_FAST=1` reduces the backtest size and (optionally) the number of channels for a faster end-to-end run.


In [ ]:
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
import os

FAST_MODE = os.getenv('CHRONOS2_TUTORIAL_FAST', '0') == '1'
USE_CUDA = torch.cuda.is_available()

# Forecast settings
FREQ = '1h'
HORIZON = 24  # next 24 hours
N_BACKTEST_WINDOWS = 3 if (FAST_MODE or not USE_CUDA) else 14

# Data locations
DATA_DIR = Path('data/raw/jena_climate')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Public mirror used by Keras examples (zip CSV)
JENA_ZIP_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip'
ZIP_PATH = DATA_DIR / 'jena_climate_2009_2016.csv.zip'
CSV_PATH = DATA_DIR / 'jena_climate_2009_2016.csv'

print(f'FAST_MODE={FAST_MODE} | USE_CUDA={USE_CUDA} | windows={N_BACKTEST_WINDOWS} | horizon={HORIZON}')


## 1. Load Jena Climate (real multivariate sensor data)

We download a zipped CSV and resample to **hourly** frequency. The dataset contains many correlated channels (temperature, pressure, humidity, wind, etc.).


In [ ]:
import urllib.request
import zipfile

if not ZIP_PATH.exists():
    print(f'Downloading: {JENA_ZIP_URL}')
    urllib.request.urlretrieve(JENA_ZIP_URL, ZIP_PATH)

if not CSV_PATH.exists():
    print(f'Extracting {ZIP_PATH.name} -> {CSV_PATH.name}')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        names = zf.namelist()
        if len(names) != 1 or not names[0].lower().endswith('.csv'):
            raise ValueError(f'Unexpected zip contents: {names}')
        zf.extract(names[0], path=DATA_DIR)
        extracted = DATA_DIR / names[0]
        extracted.rename(CSV_PATH)

print('CSV:', CSV_PATH, 'bytes:', CSV_PATH.stat().st_size)


In [ ]:
df = pd.read_csv(CSV_PATH)

# The canonical column name is "Date Time" in the Keras mirror.
if 'Date Time' not in df.columns:
    raise ValueError(f'Expected a "Date Time" column, got: {df.columns.tolist()[:10]} ...')

df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S', errors='raise')
df = df.sort_values('Date Time').drop_duplicates('Date Time').set_index('Date Time')

# Keep numeric sensor channels only.
df = df.apply(pd.to_numeric, errors='coerce')
df = df.replace([np.inf, -np.inf], np.nan)

# Hourly resample for a fair 24-step horizon (24 hours).
df_hourly = df.resample(FREQ).mean()

# Drop columns entirely missing after resampling, then forward-fill small gaps.
df_hourly = df_hourly.dropna(axis=1, how='all').ffill(limit=3)

# Drop rows still missing any values (keeps the notebook simple and leakage-safe).
df_hourly = df_hourly.dropna(axis=0, how='any')

if FAST_MODE:
    # Keep a smaller recent slice for fast execution (still real data).
    df_hourly = df_hourly.iloc[-24 * 90 :].copy()

print('Hourly rows:', len(df_hourly), '| channels:', df_hourly.shape[1])
print('Range:', df_hourly.index.min(), '->', df_hourly.index.max())
df_hourly.head()


## 2. Leakage-safe split and scaling

Chronos-2 can handle scale differences, but multivariate inference can be dominated by high-variance channels.
We standardize each channel using **train-region** statistics only (no test leakage).


In [ ]:
target_cols = df_hourly.columns.tolist()

# Optional speed knob: keep all channels in normal mode, but cap in FAST mode.
if FAST_MODE and len(target_cols) > 12:
    target_cols = target_cols[:12]
    df_hourly = df_hourly[target_cols].copy()

# Define a holdout region for rolling windows.
total_len = len(df_hourly)
needed = N_BACKTEST_WINDOWS * HORIZON
if total_len <= needed + 24 * 7:
    raise ValueError(f'Not enough data after cleaning: total_len={total_len}, needed>{needed}')

# Train region ends right before the backtest windows begin.
test_region_start = total_len - needed
train_region = df_hourly.iloc[:test_region_start]

mu = train_region.mean(axis=0)
sigma = train_region.std(axis=0).replace(0.0, 1.0)

df_scaled = (df_hourly - mu) / sigma

print('Train rows:', len(train_region), '| Eval windows:', N_BACKTEST_WINDOWS, 'x', HORIZON)
print('Target channels:', len(target_cols))


## 3. Known-future calendar covariates

We add deterministic calendar features (hour-of-day and day-of-week, encoded as sin/cos). These are **known into the future** and safe to use as future covariates.


In [ ]:
def calendar_covariates(index: pd.DatetimeIndex) -> dict[str, np.ndarray]:
    hours = index.hour.to_numpy(dtype=np.float32)
    dows = index.dayofweek.to_numpy(dtype=np.float32)

    hour_angle = 2.0 * np.pi * hours / 24.0
    dow_angle = 2.0 * np.pi * dows / 7.0

    return {
        'hour_sin': np.sin(hour_angle).astype(np.float32),
        'hour_cos': np.cos(hour_angle).astype(np.float32),
        'dow_sin': np.sin(dow_angle).astype(np.float32),
        'dow_cos': np.cos(dow_angle).astype(np.float32),
    }

# Quick sanity
_ = calendar_covariates(df_scaled.index[:5])
{k: v.shape for k, v in _.items()}


## 4. Build rolling backtest windows

Each window forecasts the next 24 hours from a different daily cutoff. We never fit or tune anything on the holdout region.


In [ ]:
backtest_windows: list[dict[str, object]] = []

for w in range(N_BACKTEST_WINDOWS):
    cutoff = test_region_start + w * HORIZON
    context_df = df_scaled.iloc[:cutoff]
    future_df = df_scaled.iloc[cutoff : cutoff + HORIZON]

    context_target = context_df[target_cols].to_numpy(dtype=np.float32).T  # (n_variates, history_len)
    future_target = future_df[target_cols].to_numpy(dtype=np.float32).T    # (n_variates, horizon)

    past_cov = calendar_covariates(context_df.index)
    future_cov = calendar_covariates(future_df.index)

    backtest_windows.append({
        'context_df': context_df,
        'future_df': future_df,
        'context_target': context_target,
        'future_target': future_target,
        'past_cov': past_cov,
        'future_cov': future_cov,
    })

print('Built windows:', len(backtest_windows))
print('Example shapes:', backtest_windows[0]['context_target'].shape, backtest_windows[0]['future_target'].shape)


## 5. Baselines and metrics

We benchmark against:

- **Naive**: repeat last value
- **Seasonal naive**: repeat the last 24-hour pattern (t-24)

Metrics are averaged across **windows and channels**.


In [ ]:
def mae(actual: np.ndarray, pred: np.ndarray) -> float:
    return float(np.mean(np.abs(actual - pred)))


def smape(actual: np.ndarray, pred: np.ndarray, eps: float = 1e-8) -> float:
    denom = np.abs(actual) + np.abs(pred) + eps
    return float(np.mean(200.0 * np.abs(actual - pred) / denom))


def evaluate(preds: list[np.ndarray], label: str) -> dict[str, float | str]:
    maes: list[float] = []
    smapes: list[float] = []
    for win, pred in zip(backtest_windows, preds):
        actual = np.asarray(win['future_target'], dtype=np.float32)
        pred = np.asarray(pred, dtype=np.float32)
        if actual.shape != pred.shape:
            raise ValueError(f'Shape mismatch: actual={actual.shape} pred={pred.shape}')
        maes.append(mae(actual, pred))
        smapes.append(smape(actual, pred))
    return {'method': label, 'MAE': float(np.mean(maes)), 'sMAPE': float(np.mean(smapes))}


naive_preds: list[np.ndarray] = []
seasonal_preds: list[np.ndarray] = []

for win in backtest_windows:
    ctx = np.asarray(win['context_target'], dtype=np.float32)
    # Naive: repeat last observation.
    last = ctx[:, -1:]  # (n_variates, 1)
    naive_preds.append(np.repeat(last, repeats=HORIZON, axis=1))

    # Seasonal naive: last 24 hours (assumes hourly frequency and horizon=24).
    seasonal_preds.append(ctx[:, -HORIZON:])

baseline_results = [
    evaluate(naive_preds, 'Naive (last value)'),
    evaluate(seasonal_preds, 'Seasonal-naive (t-24)'),
]

pd.DataFrame(baseline_results).set_index('method').round(4)


## 6. Chronos-2 multivariate inference

We compare two Chronos-2 modes:

1. **Pure multivariate target** (no covariates): list of 2D arrays `(n_variates, history_len)`
2. **Multivariate + known-future covariates**: list of dicts with `target`, `past_covariates`, `future_covariates`


In [ ]:
import time
from chronos import BaseChronosPipeline, Chronos2Pipeline

if USE_CUDA:
    pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained('amazon/chronos-2', device_map='cuda')
else:
    pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained('amazon/chronos-2')

cfg = pipeline.model.config.chronos_config
print('context_length:', cfg['context_length'])
print('max horizon:', cfg['max_output_patches'] * cfg['output_patch_size'])


In [ ]:
QUANTILE_LEVELS = [0.1, 0.5, 0.9]
BATCH_SIZE = 8 if (FAST_MODE or not USE_CUDA) else 32

# (A) Pure multivariate target
inputs_mv = [win['context_target'] for win in backtest_windows]

t0 = time.time()
q_mv, mean_mv = pipeline.predict_quantiles(
    inputs_mv, prediction_length=HORIZON, quantile_levels=QUANTILE_LEVELS, batch_size=BATCH_SIZE
)
elapsed_mv = time.time() - t0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# (B) Multivariate + calendar covariates
inputs_cov = [
    {
        'target': win['context_target'],
        'past_covariates': win['past_cov'],
        'future_covariates': win['future_cov'],
    }
    for win in backtest_windows
]

t0 = time.time()
q_cov, mean_cov = pipeline.predict_quantiles(
    inputs_cov, prediction_length=HORIZON, quantile_levels=QUANTILE_LEVELS, batch_size=BATCH_SIZE
)
elapsed_cov = time.time() - t0
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f'Chronos-2 multivariate (no covariates): {elapsed_mv:.1f}s')
print(f'Chronos-2 multivariate + calendar covariates: {elapsed_cov:.1f}s')

# Convert forecasts into numpy arrays aligned with our window loop.
preds_mv = [t.detach().cpu().numpy() if torch.is_tensor(t) else np.asarray(t) for t in mean_mv]
preds_cov = [t.detach().cpu().numpy() if torch.is_tensor(t) else np.asarray(t) for t in mean_cov]

results = baseline_results + [
    evaluate(preds_mv, 'Chronos-2 multivariate'),
    evaluate(preds_cov, 'Chronos-2 multivariate + calendar covariates'),
]

pd.DataFrame(results).set_index('method').round(4)


## 7. Visual sanity check on a few channels

We plot a single backtest window for a few selected sensor channels. The model forecasts are in **standardized units** (z-scores).


In [ ]:
import matplotlib.pyplot as plt

# Choose a few interpretable channels if present; otherwise take the first 4.
preferred = ['T (degC)', 'p (mbar)', 'rh (%)', 'wv (m/s)']
plot_cols = [c for c in preferred if c in target_cols]
if len(plot_cols) < 4:
    plot_cols = (plot_cols + [c for c in target_cols if c not in plot_cols])[:4]

w = 0 if N_BACKTEST_WINDOWS == 1 else (N_BACKTEST_WINDOWS - 1)
win = backtest_windows[w]
ctx_df = win['context_df']
fut_df = win['future_df']

pred_mv = preds_mv[w]      # (n_variates, horizon)
pred_cov = preds_cov[w]    # (n_variates, horizon)

# Quantiles: each is (n_variates, horizon, n_q)
q_cov_w = q_cov[w].detach().cpu().numpy() if torch.is_tensor(q_cov[w]) else np.asarray(q_cov[w])

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.ravel(), plot_cols):
    i = target_cols.index(col)
    ctx_tail = ctx_df[col].tail(24 * 5)
    fut = fut_df[col]

    ax.plot(ctx_tail.index, ctx_tail.values, color='black', lw=1.2, label='history (scaled)')
    ax.plot(fut.index, fut.values, color='black', lw=1.6, ls='--', label='actual (scaled)')

    ax.plot(fut.index, pred_mv[i], color='#9ca3af', lw=1.6, label='Chronos-2 mv')
    ax.plot(fut.index, pred_cov[i], color='#1a56db', lw=1.8, label='Chronos-2 mv + cal')

    lo = q_cov_w[i, :, 0]
    hi = q_cov_w[i, :, 2]
    ax.fill_between(fut.index, lo, hi, color='#1a56db', alpha=0.15, label='10-90% (cal)')

    ax.set_title(col)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=30)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=8)
fig.suptitle('Chronos-2 on Jena Climate (scaled units): multivariate vs multivariate+calendar covariates', y=1.02)
plt.tight_layout()
plt.show()


## What to do next

- Try different horizons (48h) and see whether calendar covariates help more
- Replace calendar covariates with a richer set (month-of-year, holiday flags)
- Compare Chronos-2 vs a classical multivariate baseline (VAR) on a smaller channel subset
